# `HaloGTF` `v1.0` demo

## Import

In [ ]:
from pathlib import Path
import numpy as np
import halogtf as gtf
from halogtf.io.read import load_snapshot_bundle, extract_snapshot_indices


## Instantiate config object

In [ ]:
config = gtf.Config()
config.scatter.model = "rutherford"  # You can choose "moller" (Møller scattering) too.
config.scatter.w = 30.0  # km/s; try 100.0 for weaker velocity dependence, or float("inf") for hard spheres.
config.io.base_dir = str(Path.cwd() / "demo_outputs")


This holds all of the parameters that are used to instantiate a State object. These can be modified before instantiating the State. Once the State object exists, the config object becomes an attribute of the state.

The current parameter groups are `init`, `grid`, `scatter`, `num`, `stop`, `io`, and `cosmo`. Scattering parameters live in `config.scatter` (formerly `sim`), numerical tolerances in `config.num` (formerly `prec`), and stopping limits in `config.stop`. The default `scatter.smfp_order=2`, `scatter.alph=0.57`, and `scatter.c=0.86` select the adopted Rutherford second-order calibration; choosing Møller does not automatically choose a separately calibrated parameter pair.


In [ ]:
config

Change to a truncated NFW profile, optionally modify parameters.

Numerical tolerances can affect convergence depending on the initial profile. The energy-change tolerance is `config.num.eps_du` (default `1e-4`); values such as `1e-3` require convergence checks for the chosen setup.

It is specifically the **truncated-NFW initialization** that can take a while. Below we load a precomputed initial-condition file instead. Prepare `ics/truncated_nfw_M3e9_c20_Zt005938.dat` relative to the notebook's working directory using the configuration below and the default grid. If you change the halo or grid parameters, use a matching IC file.


In [ ]:
config.init = ("truncated_nfw", {
    "Mvir": 3.0e9,  # Msun/h
    "cvir": 20.0,
    "Zt": 0.05938,
    "deltaP": 1.0e-5,
})

config


The model number is automatically chosen as the first unused number in `config.io.base_dir` when first accessed. This can be overridden by specifying `model_no`. The examples below explicitly use Model00000, Model00001, and Model00002 under `demo_outputs`; with the default `config.io.overwrite=True`, rerunning initialization can overwrite their existing outputs. Set `overwrite=False` to reject existing model directories.


In [ ]:
config.io.model_no = 0

# Instantiate the state

Construct the state from the config object, optionally loading an initial profile with `ic_filepath`. Here we use the truncated-NFW IC file prepared separately; this cell does not create it. Loading it starts a new simulation and still performs hydrostatic relaxation. `State.from_dir()` is not implemented; reading existing output files is demonstrated below instead.


In [ ]:
ic_filepath = Path.cwd() / "ics" / "truncated_nfw_M3e9_c20_Zt005938.dat"
state = gtf.State.from_config(config, ic_filepath=str(ic_filepath))
model_dir = Path(config.io.base_dir) / config.io.model_dir


At this point, a `model_metadata.txt` file has been written to the disk, as well as initial profiles.

Plot the current state (initial state):

In [ ]:
gtf.plot_snapshots(state, grid=True)

These, as well as all other plots, can be optionally saved to disk using the `filepath` keyword.

`show=True` prints the plot to stdout even when saving to a file.

In [ ]:
gtf.plot_snapshots(state, grid=True, profiles=['rho', 'v2', 'kn'],
                   filepath=str(model_dir / 'initial.png'), show=True)


# Run simulation

In each integration step:
- `dt` is determined based on the current state and tolerance parameters set in `config`.
- "Conduction step": the internal energy is updated based on the luminosity at the bin edges.  This is done to linear order (heat is only conducted in adjacent bins).  This updates `u`, and in turn, `p` (asserting $p_i = (2/3) u_i * \rho_i$)
- "Revirialization step": Now that `p` has been updated with $\rho$ unchanged, hydrostatic equilibrium must be reestablished by adjusting the edges of the radial bin, keeping `M` invariant (Lagrangian framework), but updating `rho` and `p`.
- `v2`, `u` and other derived quantities are updated based on the results of the revirialization step.


By default, the system evolves until one of three stop conditions occur:
- innermost-cell density exceeds `config.stop.rho0_halt` (default `1e5` in `rho_s` units), checked only after `state.t > 50`
- time reaches `config.stop.t_halt` (default `1000` in `t_s` units)
- central density becomes a nan

Additional conditions can be added with keyword arguments into `run()`: `steps`, `stoptime`, `rho0`

`run(stoptime=...)` requests an additional duration in `t_s` units, while `run(rho0=...)` checks the innermost-cell density immediately after each step. Saved `rho_c` is instead the mean density inside the half-central-density radius. Time uses the current reference-moment scale `t_s`, based on a common dispersion `0.33*v_s`; use the `time_Gyr` output column for physical-time comparisons. The `run()` call below retains the full-run example; you can pass `steps=100` for a short check.


In [ ]:
state.run()

If a simulation has already been fully or partially completed, we can also read a snapshot from its directory of output files:

Use the optional `snapshot` keyword to select a saved index other than the latest one. This returns a dictionary of saved data, not a restartable State; the live `state` above is retained for the plotting examples.


In [ ]:
snapshot = load_snapshot_bundle(str(model_dir))  # Latest saved snapshot
initial_snapshot = load_snapshot_bundle(str(model_dir), snapshot=0)


# Inspect/present results

In [ ]:
state.plot_time_evolution(grid=True)  # Default: rho0, the innermost-cell density
state.plot_time_evolution(grid=True, quantity='kn_cond_c')


`plot_time_evolution` also exists as a standalone function and can be used to compare multiple simulations in the same directory:

Here, Model00000 is the truncated NFW profile evolved above, Model00001 is an alpha-beta-gamma profile, and Model00002 is an NFW profile. The next cell creates and evolves the latter two before plotting, so it does not depend on the repository's legacy example outputs.

The examples use `w=30`, `100`, and `inf` km/s respectively to illustrate stronger velocity dependence, weaker velocity dependence, and the hard-sphere limit for these halo scales; these are illustrative choices, not three separately fitted calibrations. Because both the initial profiles and scattering laws differ, this is not a controlled comparison of `w` alone. The horizontal axis is each model's own `t_s` unit, and `quantity='rho_c'` plots mean core density rather than central density.


In [ ]:
# Additional initial profiles from the original comparison example.
config_abg = gtf.Config(
    init="abg",
    scatter={"model": "rutherford", "w": 100.0},  # "moller" is also supported.
    io={"base_dir": config.io.base_dir, "model_no": 1},
)
state_abg = gtf.State.from_config(config_abg)
state_abg.run()

config_nfw = gtf.Config(
    init="nfw",
    scatter={"model": "rutherford", "w": float("inf")},  # Hard-sphere limit; identical for "moller".
    io={"base_dir": config.io.base_dir, "model_no": 2},
)
state_nfw = gtf.State.from_config(config_nfw)
state_nfw.run()

gtf.plot_time_evolution(
    models=[0,1,2],
    quantity='rho_c',
    grid=True,
    base_dir=config.io.base_dir,
    filepath=str(Path(config.io.base_dir) / 'compare.png'),
    show=True
    )


When plotting snapshots, the index `-1` corresponds to the last snapshot. For the multiple-snapshot example, select indices from `snapshot_conversion.txt` so that every requested snapshot exists even if the run is short.


In [ ]:
state.plot_snapshots(grid=True, snapshots=[0,-1])

In [ ]:
available = extract_snapshot_indices(str(model_dir))['index']
selection = np.linspace(0, len(available) - 1, min(6, len(available)), dtype=int)
snapshots = [int(available[i]) for i in selection]

state.plot_snapshots(grid=True, snapshots=snapshots, profiles=['rho', 'v2', 'kn'])


# Animate the evolution

This example requires `ffmpeg` to be installed and available on your PATH. Add `parallel=False` to render frames serially if needed in your notebook environment.


In [ ]:
state.make_movie(grid=True, profiles=['rho', 'v2', 'kn'])

`make_movie()` and `plot_snapshots()` also exist as standalone functions and not only as State methods.  See, e.g., `help(gtf.make_movie)` for usage.

In [ ]:
help(gtf.make_movie), help(gtf.plot_snapshots)